<a href="https://colab.research.google.com/github/lkbrennan97/IMT-542-I3-Lightweight-App/blob/main/Colab_IMT542_I3_Lightweight_App_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Install specific libraries if not already in Colab
!pip install requests pandas plotly python-dotenv musicbrainzngs

import requests
import pandas as pd
import plotly.express as px
import json
import time
import musicbrainzngs
from IPython.display import display # For displaying DataFrames

pd.set_option('display.max_columns', None)

from google.colab import userdata

# Retrieve credentials
API_KEY = userdata.get('LastFMAPI')
SHARED_SECRET = userdata.get('LastFMSecret') # Not directly used in current code, but kept for completeness

# Last.fm API base URL and User Agent
BASE_URL = 'http://ws.audioscrobbler.com/2.0/'
USER_AGENT = 'ColabApp/1.0 (lbrenna@uw.edu)'

# MusicBrainz Setup
musicbrainzngs.set_useragent(
    "UW-MSIM-StudentProject",
    "1.0",
    "lbrenna@uw.edu"
)

def get_top_artists(limit=50):
    params = {
        'method': 'chart.gettopartists',
        'api_key': API_KEY,
        'format': 'json',
        'limit': limit
    }
    
    response = requests.get(BASE_URL, params=params, headers={'User-Agent': USER_AGENT})
    
    if response.status_code == 200:
        data = response.json()
        artists = data['artists']['artist']
        
        # Flatten the JSON into a DataFrame
        df = pd.DataFrame(artists)
        
        # Keep only the columns we need for the visualization
        df = df[['name', 'playcount', 'listeners', 'mbid', 'url']]
        
        # Convert counts to integers for plotting
        df['playcount'] = pd.to_numeric(df['playcount'])
        df['listeners'] = pd.to_numeric(df['listeners'])
        
        return df
    else:
        print(f"Error: {response.status_code}")
        return None

# Run the function to get top_artists_df
top_artists_df = get_top_artists(50)
display(top_artists_df.head())

In [1]:
# Plotting code for Top Artists by Playcount
fig = px.bar(top_artists_df.sort_values(by='playcount', ascending=False),
             x='name',
             y='playcount',
             title='Top Artists by Playcount',
             labels={'name': 'Artist', 'playcount': 'Total Playcount'})
fig.show()

NameError: name 'px' is not defined

In [ ]:
def get_artist_country(mbid, artist_name):
    try:
        if mbid:
            # Method 1: Direct lookup using MBID
            result = musicbrainzngs.get_artist_by_id(mbid)
        else:
            # Method 2: Search by name if MBID is missing
            search_result = musicbrainzngs.search_artists(artist=artist_name, limit=1)
            # Ensure there's a result before trying to access it
            if search_result['artist-list']:
                result = {'artist': search_result['artist-list'][0]}
            else:
                return "Unknown", "Unknown" # No artist found by name

        # Extract the country (usually a 2-letter code like 'US' or 'GB')
        country = result['artist'].get('country', 'Unknown')

        # Optional: Get the more specific 'area' name if you want to map cities
        area = result['artist'].get('area', {}).get('name', 'Unknown')

        return country, area
    except Exception as e:
        # print(f"Error fetching country for {artist_name} (MBID: {mbid}): {e}") # Uncomment for debugging
        return "Unknown", "Unknown"

In [ ]:
import time
from IPython.display import display # Import display for showing dataframes

## Sleep Timer and Country Call

countries = []
areas = []

print("Starting MusicBrainz Lookup (This will take ~1 minute for 50 artists)...")

for index, row in top_artists_df.iterrows():
    c, a = get_artist_country(row['mbid'], row['name'])
    countries.append(c)
    areas.append(a)
    time.sleep(1.1)  # Respect the 1-call-per-second rule

top_artists_df['country_code'] = countries
top_artists_df['area_name'] = areas

print("Mapping Complete!")
display(top_artists_df)